## Proof of concept - Claims and Payments by Policy Segment
### QUESTION:
How do claims and payments trend across different policy segments?
### INTERPRETATION:
What is a policy segment? This is different than a policy_type?
The only other pieces of data I have are state and industry, so I will cross those data points together to make visuals for all combinations. The use of "trend" makes me think the user may want to see a trend/regression line, so I will include those for both payments and claims.


### By Policy Type

In [ ]:
%%sql -r claims_payments_policy_type
select
    yearmonth,
    policy_type,
    sum(total_payments) as t_payments,
    sum(claims) as t_claims
from rli.present.prs_claims_payments_by_segment_yearmonth
group by
    yearmonth,
    policy_type;

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df = claims_payments_policy_type.sort_values('YEARMONTH')
policy_types = sorted(df['POLICY_TYPE'].unique())
n = len(policy_types)

fig, axes = plt.subplots(n, 1, figsize=(14, 5 * n))
fig.suptitle('Claims vs Payments by Year-Month', fontsize=14, fontweight='bold')

for i, policy_type in enumerate(policy_types):
    ax = axes[i]
    group = df[df['POLICY_TYPE'] == policy_type].groupby('YEARMONTH').sum().reset_index()
    yearmonths = group['YEARMONTH'].values
    x = np.arange(len(yearmonths))
    width = 0.35

    ax.bar(x - width/2, group['T_CLAIMS'].values, width, label='T_CLAIMS', alpha=0.7)
    ax.bar(x + width/2, group['T_PAYMENTS'].values, width, label='Payments', alpha=0.7)

    claims_coef = np.polyfit(x, group['T_CLAIMS'].values, 1)
    ax.plot(x, np.poly1d(claims_coef)(x), color='tab:blue', linestyle='--', linewidth=2, label='Claims Trend')

    payments_coef = np.polyfit(x, group['T_PAYMENTS'].values, 1)
    ax.plot(x, np.poly1d(payments_coef)(x), color='tab:orange', linestyle='--', linewidth=2, label='Payments Trend')

    ax.set_xlabel('Year-Month')
    ax.set_ylabel('Amount')
    ax.set_title(policy_type)
    ax.set_xticks(x)
    ax.set_xticklabels([str(ym) for ym in yearmonths], rotation=45, ha='right')
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.show()

### By State

In [ ]:
%%sql -r claims_payments_state
select
    yearmonth,
    state,
    sum(total_payments) as t_payments,
    sum(claims) as t_claims
from rli.present.prs_claims_payments_by_segment_yearmonth
group by
    yearmonth,
    state;

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df = claims_payments_state.sort_values('YEARMONTH')
states = sorted(df['STATE'].unique())
n = len(states)

fig, axes = plt.subplots(n, 1, figsize=(14, 5 * n))
fig.suptitle('Claims vs Payments by Year-Month', fontsize=14, fontweight='bold')

for i, STATE in enumerate(states):
    ax = axes[i]
    group = df[df['STATE'] == STATE].groupby('YEARMONTH').sum().reset_index()
    yearmonths = group['YEARMONTH'].values
    x = np.arange(len(yearmonths))
    width = 0.35

    ax.bar(x - width/2, group['T_CLAIMS'].values, width, label='T_CLAIMS', alpha=0.7)
    ax.bar(x + width/2, group['T_PAYMENTS'].values, width, label='Payments', alpha=0.7)

    claims_coef = np.polyfit(x, group['T_CLAIMS'].values, 1)
    ax.plot(x, np.poly1d(claims_coef)(x), color='tab:blue', linestyle='--', linewidth=2, label='Claims Trend')

    payments_coef = np.polyfit(x, group['T_PAYMENTS'].values, 1)
    ax.plot(x, np.poly1d(payments_coef)(x), color='tab:orange', linestyle='--', linewidth=2, label='Payments Trend')

    ax.set_xlabel('Year-Month')
    ax.set_ylabel('Amount')
    ax.set_title(STATE)
    ax.set_xticks(x)
    ax.set_xticklabels([str(ym) for ym in yearmonths], rotation=45, ha='right')
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.show()

### By Industry

In [ ]:
%%sql -r claims_payments_industry
select
    yearmonth,
    industry,
    sum(total_payments) as t_payments,
    sum(claims) as t_claims
from rli.present.prs_claims_payments_by_segment_yearmonth
group by
    yearmonth,
    industry;

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df = claims_payments_industry.sort_values('YEARMONTH')
industries = sorted(df['INDUSTRY'].unique())
n = len(industries)

fig, axes = plt.subplots(n, 1, figsize=(14, 5 * n))
fig.suptitle('Claims vs Payments by Year-Month', fontsize=14, fontweight='bold')

for i, INDUSTRY in enumerate(industries):
    ax = axes[i]
    group = df[df['INDUSTRY'] == INDUSTRY].groupby('YEARMONTH').sum().reset_index()
    yearmonths = group['YEARMONTH'].values
    x = np.arange(len(yearmonths))
    width = 0.35

    ax.bar(x - width/2, group['T_CLAIMS'].values, width, label='T_CLAIMS', alpha=0.7)
    ax.bar(x + width/2, group['T_PAYMENTS'].values, width, label='Payments', alpha=0.7)

    claims_coef = np.polyfit(x, group['T_CLAIMS'].values, 1)
    ax.plot(x, np.poly1d(claims_coef)(x), color='tab:blue', linestyle='--', linewidth=2, label='Claims Trend')

    payments_coef = np.polyfit(x, group['T_PAYMENTS'].values, 1)
    ax.plot(x, np.poly1d(payments_coef)(x), color='tab:orange', linestyle='--', linewidth=2, label='Payments Trend')

    ax.set_xlabel('Year-Month')
    ax.set_ylabel('Amount')
    ax.set_title(INDUSTRY)
    ax.set_xticks(x)
    ax.set_xticklabels([str(ym) for ym in yearmonths], rotation=45, ha='right')
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.show()

### By Policy Type and State
NOTE: The visuals are starting to become too large, so I'm only looking at the best and worst 5 segments by payments - claims.

In [ ]:
with ranked as (
    select
        policy_type,
        state,
        sum(total_payments) - sum(claims) as net_diff
    from rli.present.prs_claims_payments_by_segment_yearmonth
    group by policy_type, state
),
top5 as (
    select policy_type, state from ranked order by net_diff desc limit 5
),
bottom5 as (
    select policy_type, state from ranked order by net_diff asc limit 5
),
top_bottom as (
    select * from top5
    union all
    select * from bottom5
)
select
    a.yearmonth,
    a.policy_type,
    a.state,
    sum(a.total_payments) as t_payments,
    sum(a.claims) as t_claims
from rli.present.prs_claims_payments_by_segment_yearmonth a
inner join top_bottom b
    on a.policy_type = b.policy_type and a.state = b.state
group by
    a.yearmonth,
    a.policy_type,
    a.state
order by
    a.yearmonth, a.policy_type, a.state;

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df = cp_policy_type_state.sort_values('YEARMONTH')
df['SEGMENT'] = df['POLICY_TYPE'] + ' - ' + df['STATE']
segments = sorted(df['SEGMENT'].unique())
n = len(segments)

fig, axes = plt.subplots(n, 1, figsize=(14, 5 * n))
fig.suptitle('Claims vs Payments by Policy Type & State', fontsize=14, fontweight='bold')

if n == 1:
    axes = [axes]

for i, segment in enumerate(segments):
    ax = axes[i]
    group = df[df['SEGMENT'] == segment].groupby('YEARMONTH').sum(numeric_only=True).reset_index()
    yearmonths = group['YEARMONTH'].values
    x = np.arange(len(yearmonths))
    width = 0.35

    ax.bar(x - width/2, group['T_CLAIMS'].values, width, label='Claims', alpha=0.7)
    ax.bar(x + width/2, group['T_PAYMENTS'].values, width, label='Payments', alpha=0.7)

    claims_coef = np.polyfit(x, group['T_CLAIMS'].values, 1)
    ax.plot(x, np.poly1d(claims_coef)(x), color='tab:blue', linestyle='--', linewidth=2, label='Claims Trend')

    payments_coef = np.polyfit(x, group['T_PAYMENTS'].values, 1)
    ax.plot(x, np.poly1d(payments_coef)(x), color='tab:orange', linestyle='--', linewidth=2, label='Payments Trend')

    ax.set_xlabel('Year-Month')
    ax.set_ylabel('Amount')
    ax.set_title(segment)
    ax.set_xticks(x)
    ax.set_xticklabels([str(ym) for ym in yearmonths], rotation=45, ha='right')
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.show()

NOTE: I'm going to end this here. It's easy to keep copying the logic for each combination, but I think this demonstrates the thought.